# Theseus 教程（中文翻译版）

- 原始英文版：`02_differentiating_theseus_layer.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


<h1>通过Theseus层进行区分</h1>

本教程展示了如何通过Theseus 层进行微分来解决一系列相关的最小二乘优化问题。

教程 1 的优化问题是通过Theseus 非线性最小二乘优化器的一个应用程序（每个）来完成的，因为它们是简单的曲线拟合问题。 Theseus 还可用于解决更复杂的优化问题，例如，正在优化的数量之间存在依赖性。在本教程中，我们将解决一组共享一个公共参数的曲线拟合问题。与教程 1 一样，为了简单起见，我们选择二次函数：我们希望拟合 <i>y = ax<sup>2</sup> + b</i>，其中 <i>a</i> 对于所有问题都是固定的，而 <i>b</i> 对于每个问题都是不同的。

在高层次上，我们通过使用torch自动微分来优化<i>a</i>的值来解决这个问题，并使用Theseus中的非线性最小二乘优化器针对给定的<i>a</i>优化<i>b</i>。本笔记本的其余部分详细介绍了必要的步骤。

<h2>第0步：数据生成</h2>

和以前一样，我们首先通过从一组二次函数 <i>3x<sup>2</sup> + b</i> 中采样点来生成数据，其中 <i>b = 3, 5, ..., 21</i>。为此，我们添加高斯噪声 <i>&sigma; = 0.01。

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

def generate_data(num_points=100, a=1, b=0.5, noise_factor=0.01):
    # Generate data: 100 points sampled from the quadratic curve listed above
    data_x = torch.rand((1, num_points))
    noise = torch.randn((1, num_points)) * noise_factor
    data_y = a * data_x.square() + b + noise
    return data_x, data_y

def generate_learning_data(num_points, num_models):
    a, b = 3, 1
    data_batches = []
    for i in range(num_models):
        b = b + 2
        data = generate_data(num_points, a, b)
        data_batches.append(data)
    return data_batches

num_models = 10
data_batches = generate_learning_data(100, num_models)

fig, ax = plt.subplots()
for i in range(num_models):
    ax.scatter(data_batches[i][0], data_batches[i][1])
ax.set_xlabel('x');
ax.set_ylabel('y');

<h2>第1步：设置Theseus优化</h2>

接下来，我们设置与教程 1 类似的Theseus 优化问题，但有一个关键更改：<i>a</i> 不再是Theseus NLLS 优化器的优化变量；相反，它是一个辅助变量，其值将由 PyTorch 通过反向传播进行优化。 <i>b</i> 仍然是Theseus NLLS 优化器的唯一优化变量。下面的代码说明了这一点。

In [ ]:
import theseus as th

data_x, data_y = data_batches[0]
x = th.Variable(data_x, name="x")
y = th.Variable(data_y, name="y")
a = th.Vector(1, name="a")
b = th.Vector(1, name="b")

# Note 'b' is the only optim_var, and 'a' is part of aux_vars
optim_vars = [b]
aux_vars = a, x, y

# updated error function reflects change in 'a'
def quad_error_fn2(optim_vars, aux_vars):
    [b] = optim_vars 
    a, x, y = aux_vars
    est = a.tensor * x.tensor.square() + b.tensor
    err = y.tensor - est
    return err

cost_function = th.AutoDiffCostFunction(
    optim_vars, quad_error_fn2, 100, aux_vars=aux_vars, name="quadratic_cost_fn"
)
objective = th.Objective()
objective.add(cost_function)
optimizer = th.GaussNewton(
    objective,
    max_iterations=50,
    step_size=0.5,
)
theseus_optim = th.TheseusLayer(optimizer)

# The value for Variable 'a' is optimized by PyTorch backpropagation
a_tensor = torch.nn.Parameter(torch.rand(1, 1))
model_optimizer = torch.optim.Adam([a_tensor], lr=0.1)

<h2>第2步：运行优化和学习</h2>

最后，我们通过使用Theseus NLLS 优化器的学习循环来计算 <i>a</i> 和 <i>b</i>，每个模型的数据作为一个批次。 Theseus NLLS 优化器针对当前 `a` 优化 `b`（称为内循环优化），而 PyTorch 反向传播学习跨批次使用 `a` 的正确值（称为外循环）优化</i>）。

为了清楚起见，我们描述了所需的高级步骤：
- 步骤 2.1：与教程 1 一样，我们首先创建输入字典，并将其传递给 `TheseusLayer` 的 `forward` 函数。请注意，此处的 `forward` 函数不会跟踪本示例中的最佳解决方案，因为整个 NLLS 优化序列需要用于反向传播。 
在 `TheseusLayer` 中完成优化后，我们得到一个字典 `updated_inputs` ，其中包含优化变量的最新值（所有其他字典值不变）。 
- 步骤2.2：然后我们用`updated_inputs`字典更新`objective`，并用它来计算损失。这种对Theseus 优化变量的使用是 `forward` 返回输入字典的原因。
- 步骤 2.3：该损失用于反向传播。 PyTorch 学习优化器（即此处的 Adam）现在将对学习参数（即本示例中的 `a` 的值）采取一个优化步骤。
- 迭代：我们现在对 `forward` 进行新的调用，重复步骤 2.1-2.3。

我们在下面的代码中说明了这些步骤。

In [ ]:
num_batches = len(data_batches)
num_epochs = 20

print(f"Initial a value: {a_tensor.item()}")

for epoch in range(num_epochs):
    epoch_loss = 0.
    epoch_b = []  # keep track of the current b values for each model in this epoch
    for i in range(num_batches):
        model_optimizer.zero_grad()
        data_x, data_y = data_batches[i]
        # Step 2.1: Create input dictionary for TheseusLayer, pass to forward function
        # The value for variable `a` is the updated `a_tensor` by Adam
        # Since we are always passing the same tensor, this update is technically redundant, 
        # we include it to explicitly illustrate where variable values come from.
        # An alternative way to do this (try it!)
        # would be to call `a.update(a_tensor)` before the learning loop starts
        # and just let Adam change its value under the hood (i.e., update only `x`, `y`, and `b`
        # inside the loop)
        theseus_inputs = {
            "a": a_tensor,
            "x": data_x,
            "y": data_y,
            "b": torch.ones((1, 1)),
        }
        updated_inputs, info = theseus_optim.forward(theseus_inputs)
        epoch_b.append(updated_inputs["b"].item())  # add the optimized "b" of the current batch to `epoch_b` list. 
                                                    # Note: here, we do not track the best solution, as we 
                                                    # backpropagate through the entire optimization sequence.
        # Step 2.2: Update objective function with updated inputs
        objective.update(updated_inputs)
        loss = cost_function.error().square().mean()
        # Step 2.3: PyTorch backpropagation
        loss.backward()
        model_optimizer.step()

        loss_value = loss.item()
        epoch_loss += loss_value
    print(f"Epoch: {epoch} Loss: {epoch_loss}")
    if epoch % 5 == 4:
        print(f" ---------------- Solutions at Epoch {epoch:02d} -------------- ")
        print(" a value:", a.tensor.item())
        print(" b values: ", epoch_b)
        print(f" ----------------------------------------------------- ")

In [ ]:
# Plot the learned functions
fig, ax = plt.subplots()

for i in range(num_models):
    ax.scatter(data_batches[i][0], data_batches[i][1])

    a_ = a.tensor.squeeze().detach()
    b = epoch_b[i]
    x = torch.linspace(0., 1., steps=100)
    y = a_*x*x + b
    ax.plot(x, y, color='k', lw=4, linestyle='--',
            label='Learned quadratics' if i == 0 else None)
ax.legend()

ax.set_xlabel('x');
ax.set_ylabel('y');

我们观察到我们能够恢复 `a` 和 `b` 非常接近我们采样的值。

<h2>步骤3（可选）：同时解决所有优化问题</h2>

上述只是用Theseus 来模拟我们的问题的多种方法之一。 Theseus 还支持同时求解多个优化问题，因此我们也可以同时求解所有 10 个最小二乘优化问题。我们可以通过两种自然的方式做到这一点：
- 版本 A：通过创建 10 个 `AutoDiffCostFunction`，每个优化问题一个。在这里，我们需要每个 `AutoDiffCostFunction` 都有一个单独的 `b, x, y` 变量（例如，`[b1, x1, y1]`、`[b2, x2, y2]` 等）。所有代价函数都添加到目标中，可以与一个`forward`联合优化，并与后面的`backward`联合求导。 
- 版本 B：通过更改要批处理的 `b, x, y` 变量，并使用 `quad_err_fn2` 之上的误差函数来支持批处理，我们可以使用单个 `AutoDiffCostFunction` 来捕获其拟合成本。然而，由于错误现在可能是批量的，因此必须将损失计算为评估目标的聚合。

版本 A 更常用于每个变量和成本函数具有不同语义解释的情况（例如，教程 4 和 5 的运动规划问题中的不同时间步），而版本 B 通常用于同一问题的多个实例（例如，教程 4 和 5 中的不同地图）。我们在下面显示了每个版本的完整代码。两个版本都发现非常相似的 `a` 和 `b` 值。

由于两个版本都需要一个通用的学习例程，因此我们首先创建一个子例程 `optimize_and_learn_models_jointly` 以提高可读性。

In [ ]:
# Sub-routine to optimize and learn models simultaneously

def optimize_and_learn_models_jointly(theseus_optim, model_optimizer, num_epochs=20):
    # again, assume a_tensor is initialized outside this loop
    print(f"Initial a value: {a_tensor.item()}")
    
    for epoch in range(num_epochs):
        model_optimizer.zero_grad()
        # Step 2.1: Create input dictionary for TheseusLayer, pass to forward function
        theseus_inputs = construct_theseus_layer_inputs()
        updated_inputs, _ = theseus_optim.forward(theseus_inputs)

        # Step 2.2: Update objective function with updated inputs
        objective.update(updated_inputs)
        loss = objective.error_metric() 
        loss = loss.sum()  # Note `loss` now needs a final aggregation (for version B)

        # Step 2.3: PyTorch backpropagation
        loss.backward()
        model_optimizer.step()

        if epoch == 0:
            min_loss = loss
        if loss <= min_loss:
            min_loss = loss
            best_model_a = a.tensor.item()
            best_model_b = [b.tensor.item() for b in all_b]

        print(f"Epoch: {epoch} Loss: {loss.item()}")
        if epoch % 10 == 9:
            print(f" ---------------- Solutions at Epoch {epoch:02d} -------------- ")
            print(" a value:", a.tensor.item())
            print(" b values: ", [b.tensor.item() for b in all_b])
            print(f" ----------------------------------------------------- ")

    return best_model_a, best_model_b

<h3>步骤 3.1：示例版本 A</h3>

现在我们展示版本A。该版本对原始代码片段进行了以下更改：
- 创建 10 个 `x`、`y` 和 `b` 变量。
- 创建10个`AutoDiffCostFunctions`，每个使用相同的`a`，但使用相应的`b_i`、`x_i`和`y_i`。所有成本函数都添加到目标中。
- 使用映射到正确数据批次的 `b_i`、`x_i` 和 `y_i` 构建 `theseus_inputs` 字典。

请注意，`a` 及其 PyTorch 优化器的设置保持不变。设置优化问题后，我们调用 `optimize_and_learn_models_jointly` 子例程来计算 `a` 和 `b` 值。

In [ ]:
# Version A

# replace x and y with 10 different variables x0 ... x9, y0 ... y9
all_x, all_y = [], []
for i in range(num_models): 
    all_x.append(th.Variable(data_x, name=f"x{i}"))
    all_y.append(th.Variable(data_y, name=f"y{i}"))

# replace b with 10 different variables b0 ... b9
all_b = []
for i in range(num_models):
    all_b.append(th.Vector(1, name=f"b{i}"))

# a remains the same
a = th.Vector(1, name="a")

# objective now has 10 different cost functions
objective = th.Objective()
for i in range(num_models):
    # each cost function i uses b_i as optim_var and x_i, y_i and a as aux_var
    optim_vars = [all_b[i]]
    aux_vars = a, all_x[i], all_y[i]
    cost_function = th.AutoDiffCostFunction(
        optim_vars, quad_error_fn2, 100, aux_vars=aux_vars, name=f"quadratic_cost_fn_{i}"
    )
    objective.add(cost_function)

# optimizer, TheseusLayer and model optimizer remains the same
optimizer = th.GaussNewton(
    objective, max_iterations=50, step_size=0.4,
)
theseus_optim = th.TheseusLayer(optimizer)
a_tensor = torch.nn.Parameter(torch.rand(1, 1))
model_optimizer = torch.optim.Adam([a_tensor], lr=0.15)

# TheseusLayer dictionary now needs to construct b0 ... b9, x0 ... x9, y0 ... y9
def construct_theseus_layer_inputs():
    theseus_inputs = {}
    for i in range(num_models):
        data_x, data_y = data_batches[i]
        theseus_inputs.update({
            f"x{i}": data_x,
            f"y{i}": data_y,
            f"b{i}": torch.ones((1, 1)),
        })
    theseus_inputs.update({"a": a_tensor})
    return theseus_inputs

# Run Theseus optimization and learning
best_model = optimize_and_learn_models_jointly(theseus_optim, model_optimizer)

print(f" ---------------- Final Solutions -------------- ")
print(" a value:", best_model[0])
print(" b values: ", best_model[1])
print(f" ----------------------------------------------- ")


In [ ]:
# Plot the learned functions
fig, ax = plt.subplots()

for i in range(num_models):
    ax.scatter(data_batches[i][0], data_batches[i][1])

    a = best_model[0]
    b = best_model[1][i]
    x = torch.linspace(0., 1., steps=100)
    y = a*x*x + b
    ax.plot(x, y, color='k', lw=4, linestyle='--',
            label='Learned quadratics' if i == 0 else None)
ax.legend()

ax.set_xlabel('x');
ax.set_ylabel('y');

<h3>步骤3.2：版本B示例</h3>

现在我们展示版本B。该版本对原始代码片段进行了以下更改：
- 创建大小为 `[num_models, 100]` 的 `x`、`y` 变量，而不是 `[1, 100]`
- 创建大小为 `[num_models, 1]` 的 `b` 变量，而不是 `[1, 1]`
- 使用`b`、`x`和`y`数据批量构建`theseus_inputs`字典

在此示例中，我们不需要更改错误函数，因为 PyTorch 张量处理广播以使批处理版本正常工作。

和以前一样，`a` 及其 PyTorch 优化器的设置保持不变。一旦优化问题成立，我们调用`optimize_and_learn_models_jointly`子例程来解决问题。

In [ ]:
# Version B

# convert data_x, data_y into torch.tensors of shape [num_models, 100]
data_x = torch.stack([data_x.squeeze() for data_x, _ in data_batches])
data_y = torch.stack([data_y.squeeze() for _, data_y in data_batches])

# construct one variable each of x, y of shape [num_models, 100]
x = th.Variable(data_x, name="x")
y = th.Variable(data_y, name="y")

# construct a as before
a = th.Vector(1, name="a")

# construct one variable b, now of shape [num_models, 1]
b = th.Vector(tensor=torch.rand(num_models, 1), name="b")

# Again, 'b' is the only optim_var, and 'a' is part of aux_vars along with x, y
optim_vars = [b]
aux_vars = a, x, y

# cost function constructed as before 
cost_function = th.AutoDiffCostFunction(
    optim_vars, quad_error_fn2, 100, aux_vars=aux_vars, name="quadratic_cost_fn"
)

# objective, optimizer and theseus layer constructed as before
objective = th.Objective()
objective.add(cost_function)
optimizer = th.GaussNewton(
    objective, max_iterations=50, step_size=0.5,
)
theseus_optim = th.TheseusLayer(optimizer)

# As before, 'a' is optimized by PyTorch backpropagation
# model_optimizer constructed the same way 
a_tensor = torch.nn.Parameter(torch.rand(1, 1))
model_optimizer = torch.optim.Adam([a_tensor], lr=0.2)

# The theseus_inputs dictionary is also constructed similarly to before,
# but with data matching the new shapes of the variables
def construct_theseus_layer_inputs():
    theseus_inputs = {}
    theseus_inputs.update({
        "x": data_x,
        "y": data_y,
        "b": torch.ones((num_models, 1)),
        "a": a_tensor,
    })
    return theseus_inputs

# Run Theseus optimization and learning
best_model = optimize_and_learn_models_jointly(theseus_optim, model_optimizer)
print(f" ---------------- Final Solutions -------------- ")
print(" a value:", best_model[0])
print(" b values: ", best_model[1])
print(f" ----------------------------------------------- ")

In [ ]:
# Plot the learned functions
fig, ax = plt.subplots()

for i in range(num_models):
    ax.scatter(data_batches[i][0], data_batches[i][1])

    a = best_model[0]
    b = best_model[1][i]
    x = torch.linspace(0., 1., steps=100)
    y = a*x*x + b
    ax.plot(x, y, color='k', lw=4, linestyle='--',
            label='Learned quadratics' if i == 0 else None)
ax.legend()

ax.set_xlabel('x');
ax.set_ylabel('y');

技术说明：您可能会注意到，在此示例中，微分仅通过TheseusLayer 目标发生，而不是通过Theseus 非线性最小二乘优化器发生。这是由于示例问题的简单性。一系列更复杂的内循环计算将需要Theseus 通过非线性最小二乘优化器进行微分；此类问题位于教程 4 和 5 中，以及 [Theseus `examples` 文件夹](https://github.com/facebookresearch/theseus/tree/main/examples) 中。